# Figure 2c — candidate matching and time-course heatmaps

Run from this folder or the `P450_analysis/` root. All seven source TSV files are in `input/`: one `03_CYP_steroid_feature_annotations.tsv` and six `*_time_course_full_Feature_table_{pellets,supernatant}.tsv` files (Estradiol, DHEA, Pregnenolone).

Candidates are matched to time-course features within ±10 ppm and ±3 s, retaining every hit. Each steroid/fraction table is normalized by top-500-feature TIC. We take log2(normalized abundance + 1), average three biological replicates, then calculate each feature's row z-score across the 13 unique sample groups. The same three-replicate Control appears at D0 h and K0 h for display.

`results/` contains three concise, numbered tables: `01_candidate_matches.tsv` (every match and unmatched candidate), `02_matched_timecourse.tsv` (one row per plotted feature and displayed time point, with its 1-based plot position), and `03_processing_summary.tsv` (steroid/fraction/CYP counts and normalization range). The notebook also produces the four existing heatmap variants for CYP3A4/CYP3A5 with and without labels, each as PDF and SVG at 175 × 72 mm. The 521 plotted feature columns give 7,294 feature–time rows. Source IDs are joined with `;` when two candidates map to one plotted feature. Plot position is scoped to CYP and steroid: some time-course IDs recur under different steroids.

## 1. Read inputs and set parameters

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.font_manager import FontProperties, fontManager
from matplotlib.patches import Rectangle

HERE = Path.cwd().resolve()
if not (HERE / "input" / "03_CYP_steroid_feature_annotations.tsv").is_file():
    HERE = HERE / "Figure_modification" / "Figure_2c_time_course"
INPUT, RESULTS = HERE / "input", HERE / "results"

STEROIDS = ("Estradiol", "DHEA", "Pregnenolone")
MATRICES = ("pellets", "supernatant")
ENZYMES = ("CYP3A4", "CYP3A5")
CONDITIONS = ("DMSO", "Ketoconazole")
TIMES = ("30min", "1h", "3h", "6h", "12h", "24h")
MZ_PPM, RT_SECONDS, TOP_FEATURES = 10.0, 3.0, 500

candidate_file = INPUT / "03_CYP_steroid_feature_annotations.tsv"
feature_files = {
    (steroid, matrix): INPUT / f"{steroid}_time_course_full_Feature_table_{matrix}.tsv"
    for steroid in STEROIDS for matrix in MATRICES
}
missing = [p for p in [candidate_file, *feature_files.values()] if not p.is_file()]
if missing:
    raise FileNotFoundError(f"Missing Figure 2c inputs: {missing}")

candidates = pd.read_csv(candidate_file, sep="\t", keep_default_na=False)
required = {
    "steroid", "derivatization", "matrix", "enzyme", "unlabeled_feature",
    "unlabeled_feature_mz", "unlabeled_feature_rtime",
    "authentic_standard_identification", "parent_candidate_annotation",
}
if required - set(candidates.columns):
    raise ValueError(f"Missing candidate columns: {sorted(required - set(candidates.columns))}")
candidates = candidates.loc[
    candidates.steroid.isin(STEROIDS)
    & candidates.matrix.isin(MATRICES)
    & candidates.enzyme.isin(ENZYMES)
].copy()
if candidates.duplicated(["steroid", "matrix", "enzyme", "unlabeled_feature"]).any():
    raise ValueError("Duplicate steroid/matrix/enzyme/feature candidate")
for column in ("unlabeled_feature_mz", "unlabeled_feature_rtime"):
    candidates[column] = pd.to_numeric(candidates[column], errors="raise")
if not np.isfinite(candidates[["unlabeled_feature_mz", "unlabeled_feature_rtime"]].to_numpy()).all():
    raise ValueError("Non-finite candidate m/z or retention time")
candidates["candidate_group"] = candidates.enzyme.str.removeprefix("CYP") + "_" + candidates.matrix
candidates["source_feature_id"] = candidates.unlabeled_feature
print(f"{len(candidates)} steroid/fraction/CYP candidate rows from {candidate_file.name}")

638 steroid/fraction/CYP candidate rows from 03_CYP_steroid_feature_annotations.tsv


## 2. Normalize and match

In [2]:
def parse_sample_name(name):
    enzyme = name.split("_", 1)[0]
    if enzyme not in ("3A4", "3A5"):
        raise ValueError(f"Unexpected enzyme: {name}")
    if "_Control_" in name:
        condition, time = "Control", "Control"
    elif "_Ketoconazole_" in name or "_DMSO_" in name:
        condition = "Ketoconazole" if "_Ketoconazole_" in name else "DMSO"
        found = re.search(r"(?:^|_)(30min|1h|3h|6h|12h|24h)(?:_|$)", name)
        time = found.group(1) if found else None
    else:
        raise ValueError(f"Unexpected condition: {name}")
    found = re.search(r"_(?:P|S)_(\d)(?:_|$)", name)
    if found is None:
        found = re.search(r"_(?:30min|1h|3h|6h|12h|24h)_(\d)$", name)
    replicate = int(found.group(1)) if found else None
    if time is None or replicate is None:
        raise ValueError(f"Cannot parse sample: {name}")
    return {"sample": name, "enzyme": enzyme, "condition": condition,
            "time": time, "replicate": replicate}


def load_timecourse(steroid, matrix):
    path = feature_files[steroid, matrix]
    table = pd.read_csv(path, sep="\t", index_col="id_number")
    table.index = table.index.astype(str)
    if table.index.has_duplicates:
        raise ValueError(f"Duplicate feature IDs: {path}")
    sample_columns = [c for c in table if c.startswith(("3A4_", "3A5_"))]
    if len(sample_columns) != 78:
        raise ValueError(f"Expected 78 samples in {path}, found {len(sample_columns)}")
    metadata = pd.DataFrame([parse_sample_name(c) for c in sample_columns])
    observed = metadata.groupby(["enzyme", "condition", "time"]).size().to_dict()
    expected = {(e, "Control", "Control"): 3 for e in ("3A4", "3A5")}
    expected.update({
        (e, condition, time): 3
        for e in ("3A4", "3A5") for condition in CONDITIONS for time in TIMES
    })
    if observed != expected:
        raise ValueError(f"Unexpected sample design: {path}")
    for column in ("mz", "rtime"):
        table[column] = pd.to_numeric(table[column], errors="raise")
    if not np.isfinite(table[["mz", "rtime"]].to_numpy()).all():
        raise ValueError(f"Non-finite m/z or retention time: {path}")
    raw = table[sample_columns].apply(pd.to_numeric, errors="raise")
    if raw.isna().any().any() or (raw < 0).any().any():
        raise ValueError(f"Missing or negative intensity: {path}")
    top_ids = raw.mean(axis=1).nlargest(TOP_FEATURES).index
    top_tic = raw.loc[top_ids].sum(axis=0)
    if (top_tic <= 0).any():
        raise ValueError(f"Non-positive top-500 TIC: {path}")
    scale = top_tic / top_tic.mean()
    normalized_log2 = np.log2(raw.div(scale, axis=1) + 1)
    normalization = {
        "steroid": steroid, "matrix": matrix, "n_features": len(table),
        "n_samples": len(sample_columns), "top_features": TOP_FEATURES,
        "scale_min": scale.min(), "scale_median": scale.median(),
        "scale_mean": scale.mean(), "scale_max": scale.max(),
    }
    return table, normalized_log2, metadata, normalization

def match_group(group, table):
    tc_mz = table.mz.to_numpy(dtype=float)
    tc_rt = table.rtime.to_numpy(dtype=float)
    tc_ids = table.index.to_numpy(dtype=str)
    rows = []
    for _, candidate in group.iterrows():
        mz, rt = float(candidate.unlabeled_feature_mz), float(candidate.unlabeled_feature_rtime)
        ppm_difference = (tc_mz - mz) / mz * 1e6
        rt_difference = tc_rt - rt
        hits = np.flatnonzero(
            (np.abs(ppm_difference) <= MZ_PPM) & (np.abs(rt_difference) <= RT_SECONDS)
        )
        base = candidate.to_dict()
        if len(hits) == 0:
            rows.append({
                **base, "timecourse_feature_id": pd.NA, "timecourse_mz": np.nan,
                "timecourse_rt": np.nan, "ppm_difference": np.nan,
                "rt_difference_s": np.nan, "match_count": 0, "match_rank": pd.NA,
                "match_status": "unmatched", "match_label": "unmatched",
            })
            continue
        status = "unique" if len(hits) == 1 else "ambiguous"
        ordered = sorted(
            hits, key=lambda pos: (abs(rt_difference[pos]), abs(ppm_difference[pos]), tc_ids[pos])
        )
        for rank, pos in enumerate(ordered, start=1):
            rows.append({
                **base, "timecourse_feature_id": tc_ids[pos],
                "timecourse_mz": tc_mz[pos], "timecourse_rt": tc_rt[pos],
                "ppm_difference": ppm_difference[pos], "rt_difference_s": rt_difference[pos],
                "match_count": len(hits), "match_rank": rank,
                "match_status": status,
                "match_label": status if status == "unique" else f"ambiguous {rank}/{len(hits)}",
            })
    return pd.DataFrame(rows)


def sample_groups(metadata, enzyme):
    groups = [("Control", metadata.loc[
        (metadata.enzyme == enzyme) & (metadata.condition == "Control"), "sample"
    ].sort_values().tolist())]
    for condition in CONDITIONS:
        for time in TIMES:
            samples = metadata.loc[
                (metadata.enzyme == enzyme) & (metadata.condition == condition)
                & (metadata.time == time), "sample"
            ].sort_values().tolist()
            groups.append((f"{condition}_{time}", samples))
    if any(len(samples) != 3 for _, samples in groups):
        raise ValueError(f"Expected three replicates per group: {enzyme}")
    return groups


def build_plot_data(matches, normalized_log2, metadata, candidate_group):
    matched_ids = matches.loc[
        matches.match_status != "unmatched", "timecourse_feature_id"
    ].drop_duplicates().tolist()
    if not matched_ids:
        return pd.DataFrame(columns=[
            "candidate_group", "timecourse_feature_id", "condition", "time",
            "row_zscore", "mean_normalized_log2",
        ])
    enzyme = candidate_group.split("_", 1)[0]
    selected = normalized_log2.loc[matched_ids]
    means = pd.DataFrame({
        label: selected[samples].mean(axis=1)
        for label, samples in sample_groups(metadata, enzyme)
    }, index=matched_ids)
    row_mean = means.mean(axis=1)
    row_sd = means.std(axis=1, ddof=0).replace(0, np.nan)
    zscore = means.sub(row_mean, axis=0).div(row_sd, axis=0).fillna(0)
    records = []
    for feature_id in matched_ids:
        for condition in CONDITIONS:
            records.append({
                "candidate_group": candidate_group, "timecourse_feature_id": feature_id,
                "condition": condition, "time": "0 h",
                "row_zscore": zscore.loc[feature_id, "Control"],
                "mean_normalized_log2": means.loc[feature_id, "Control"],
            })
            for time in TIMES:
                records.append({
                    "candidate_group": candidate_group, "timecourse_feature_id": feature_id,
                    "condition": condition, "time": time,
                    "row_zscore": zscore.loc[feature_id, f"{condition}_{time}"],
                    "mean_normalized_log2": means.loc[feature_id, f"{condition}_{time}"],
                })
    result = pd.DataFrame(records)
    result.row_zscore = result.row_zscore.round(6)
    return result

In [3]:
RESULTS.mkdir(parents=True, exist_ok=True)
all_matches, all_plotted_timecourses, summaries, normalizations = [], [], [], []
plot_tables = {}
plot_columns = ["candidate_group", "timecourse_feature_id", "condition", "time", "row_zscore"]
time_minutes = {"0 h": 0, "30min": 30, "1h": 60, "3h": 180,
                "6h": 360, "12h": 720, "24h": 1440}

for steroid in STEROIDS:
    steroid_matches, steroid_plots = [], []
    for matrix in MATRICES:
        table, normalized_log2, metadata, normalization = load_timecourse(steroid, matrix)
        normalizations.append(normalization)
        for enzyme in ENZYMES:
            group_name = f"{enzyme.removeprefix('CYP')}_{matrix}"
            group = candidates.loc[
                (candidates.steroid == steroid) & (candidates.matrix == matrix)
                & (candidates.enzyme == enzyme)
            ].sort_values(["unlabeled_feature_mz", "unlabeled_feature_rtime", "source_feature_id"])
            matched = match_group(group, table)
            if len(group) != matched[["source_feature_id", "candidate_group"]].drop_duplicates().shape[0]:
                raise AssertionError(f"Candidate relations lost: {steroid}/{group_name}")
            plot = build_plot_data(matched, normalized_log2, metadata, group_name)
            steroid_matches.append(matched)
            steroid_plots.append(plot)
            status = matched.drop_duplicates(["source_feature_id", "candidate_group"]).match_status.value_counts()
            summary = {
                "steroid": steroid, "candidate_group": group_name,
                "matrix": matrix, "enzyme": enzyme, "candidate_count": len(group),
                "unique": int(status.get("unique", 0)),
                "ambiguous": int(status.get("ambiguous", 0)),
                "unmatched": int(status.get("unmatched", 0)),
                "match_rows": len(matched),
                "plotted_timecourse_features": plot.timecourse_feature_id.nunique(),
                "mz_tolerance_ppm": MZ_PPM, "rt_tolerance_seconds": RT_SECONDS,
                "normalization_top_features": TOP_FEATURES,
            }
            if sum(summary[k] for k in ("unique", "ambiguous", "unmatched")) != len(group):
                raise AssertionError(f"Bad match-status total: {steroid}/{group_name}")
            summaries.append(summary)
    matches = pd.concat(steroid_matches, ignore_index=True)
    plots = pd.concat(steroid_plots, ignore_index=True)
    if plots.duplicated(["candidate_group", "timecourse_feature_id", "condition", "time"]).any():
        raise AssertionError(f"Duplicate plot time point for {steroid}")
    matched_hits = matches.loc[matches.match_status.ne("unmatched")]
    target_key = ["candidate_group", "timecourse_feature_id"]
    if any(matched_hits.groupby(target_key)[field].nunique().gt(1).any()
           for field in ("derivatization", "matrix", "enzyme")):
        raise AssertionError(f"Conflicting target metadata for {steroid}")
    source_meta = matched_hits.groupby(target_key, sort=False).agg(
        derivatization=("derivatization", "first"),
        matrix=("matrix", "first"),
        enzyme=("enzyme", "first"),
        matched_source_feature_ids=(
            "source_feature_id", lambda ids: ";".join(pd.unique(ids.astype(str)))
        ),
    ).reset_index()
    plotted_timecourse = plots.merge(
        source_meta, on=target_key, how="left", sort=False, validate="many_to_one",
    )
    if len(plotted_timecourse) != len(plots) or plotted_timecourse.matched_source_feature_ids.isna().any():
        raise AssertionError(f"Missing plotted-feature source annotation for {steroid}")
    plotted_timecourse["steroid"] = steroid
    plotted_timecourse["time_min"] = plotted_timecourse.time.map(time_minutes)
    # One row per plotted feature and condition/time; full diagnostics are exported in 01_candidate_matches.tsv.
    integrated_columns = [
        "steroid", "derivatization", "matrix", "enzyme", "matched_source_feature_ids",
        "timecourse_feature_id", "condition", "time_min", "mean_normalized_log2", "row_zscore",
    ]
    all_plotted_timecourses.append(plotted_timecourse[integrated_columns])
    plot_tables[steroid] = plots[plot_columns].copy()
    all_matches.append(matches)

all_matches = pd.concat(all_matches, ignore_index=True)
summary = pd.DataFrame(summaries)
normalization_summary = pd.DataFrame(normalizations)
integrated = pd.concat(all_plotted_timecourses, ignore_index=True)
if len(integrated) != 7294:
    raise AssertionError(f"Expected 7294 plotted feature–time rows, got {len(integrated)}")
integrated["_enzyme_order"] = integrated.enzyme.map({name: pos for pos, name in enumerate(ENZYMES)})
integrated["_steroid_order"] = integrated.steroid.map({name: pos for pos, name in enumerate(STEROIDS)})
integrated = integrated.sort_values(["_enzyme_order", "_steroid_order"], kind="stable").drop(
    columns=["_enzyme_order", "_steroid_order"]
)
print(f"Prepared {len(all_matches)} match rows and {len(integrated)} plotted feature–time rows")


Prepared 721 match rows and 7294 plotted feature–time rows


## 3. Draw the four heatmap variants

In [4]:
# Figure 2c: original panel geometry and colors; outputs PDF/SVG only.
# Required inputs from earlier cells: RESULTS, ENZYMES, plot_tables and listed imports.


STEROID_ORDER = ["Estradiol", "DHEA", "Pregnenolone"]
STEROID_COLORS = {
    "Estradiol": "#4E79A7",
    "DHEA": "#59A14F",
    "Pregnenolone": "#F2B134",
}
FRACTION_COLORS = {
    "supernatant": "#859DB4",
    "pellets": "#B39778",
}
ENZYME_ORDER = [enzyme.removeprefix("CYP") for enzyme in ENZYMES]
CONDITION_ORDER = ["DMSO", "Ketoconazole"]
TIME_ORDER = ["0 h", "30min", "1h", "3h", "6h", "12h", "24h"]
DISPLAY_TIME_LABELS = [
    "D0 h", "D30 min", "D1 h", "D3 h", "D6 h", "D12 h", "D24 h",
    "K0 h", "K30 min", "K1 h", "K3 h", "K6 h", "K12 h", "K24 h",
]


def force_svg_arial(svg_path):
    """Keep SVG text editable while making Arial the explicit sole family."""
    svg_path = Path(svg_path)
    svg_text = svg_path.read_text(encoding="utf-8")
    svg_text, replacements = re.subn(
        r"font-family:\s*[^;]+;",
        "font-family: Arial;",
        svg_text,
    )
    if replacements == 0 and "<text" in svg_text:
        raise ValueError("SVG text has no font-family declarations.")
    svg_path.write_text(svg_text, encoding="utf-8")

# Validate the in-memory 01 plot tables before drawing.
required_plot_columns = {
    "candidate_group", "timecourse_feature_id", "condition", "time", "row_zscore",
}
if set(plot_tables) != set(STEROID_ORDER):
    raise ValueError(f"Unexpected steroid tables: {sorted(plot_tables)}")
if ENZYME_ORDER != ["3A4", "3A5"]:
    raise ValueError(f"Unexpected enzyme order: {ENZYME_ORDER}")
tables = {}
for steroid in STEROID_ORDER:
    table = plot_tables[steroid].copy()
    if set(table.columns) != required_plot_columns:
        raise ValueError(f"Unexpected plot columns for {steroid}: {table.columns.tolist()}")
    if table["timecourse_feature_id"].isna().any():
        raise ValueError(f"Missing feature IDs for {steroid}")
    table["timecourse_feature_id"] = table["timecourse_feature_id"].astype(str)
    if set(table["condition"]) != set(CONDITION_ORDER):
        raise ValueError(f"Unexpected conditions for {steroid}")
    if set(table["time"]) != set(TIME_ORDER):
        raise ValueError(f"Unexpected time points for {steroid}")
    table["row_zscore"] = pd.to_numeric(table["row_zscore"], errors="raise")
    if not np.isfinite(table["row_zscore"]).all():
        raise ValueError(f"Non-finite z-scores for {steroid}")
    tables[steroid] = table


def build_enzyme_matrix(tables, enzyme):
    """Pool pellets/supernatant and return a steroid-blocked feature matrix."""
    matrices = []
    feature_rows = []
    duplicate_rows_removed = 0

    for steroid in STEROID_ORDER:
        table = tables[steroid]
        selected = table.loc[
            table["candidate_group"].str.startswith(f"{enzyme}_")
        ].copy()
        if selected.empty:
            raise ValueError(f"No {enzyme} rows for {steroid}")

        # Preserve first appearance from the source export. Exact repeated rows
        # in the DHEA file are collapsed so each feature is drawn once.
        feature_order = selected["timecourse_feature_id"].drop_duplicates().tolist()
        cell_key = ["timecourse_feature_id", "condition", "time"]
        disagreements = (
            selected.groupby(cell_key, sort=False)["row_zscore"]
            .agg(lambda values: float(values.max() - values.min()))
        )
        if (disagreements > 1e-9).any():
            bad = disagreements.loc[disagreements > 1e-9].index.tolist()[:5]
            raise ValueError(f"Conflicting duplicate values for {steroid}/{enzyme}: {bad}")
        before = len(selected)
        selected = selected.drop_duplicates(cell_key, keep="first")
        duplicate_rows_removed += before - len(selected)

        selected["sample_order"] = pd.Categorical(
            selected["condition"] + "|" + selected["time"],
            categories=[
                f"{condition}|{time}"
                for condition in CONDITION_ORDER
                for time in TIME_ORDER
            ],
            ordered=True,
        )
        matrix = selected.pivot(
            index="sample_order", columns="timecourse_feature_id", values="row_zscore",
        ).reindex(columns=feature_order)
        matrix = matrix.reindex(selected["sample_order"].cat.categories)
        if matrix.shape != (14, len(feature_order)) or matrix.isna().any().any():
            raise ValueError(f"Incomplete 14-point profiles for {steroid}/{enzyme}")
        matrices.append(matrix)

        sources = (
            selected.groupby("timecourse_feature_id", sort=False)["candidate_group"]
            .agg(lambda values: ";".join(dict.fromkeys(values)))
        )
        for within_steroid_position, feature_id in enumerate(feature_order, start=1):
            feature_rows.append({
                "enzyme": enzyme,
                "steroid": steroid,
                "within_steroid_position": within_steroid_position,
                "timecourse_feature_id": feature_id,
                "source_candidate_group": sources.loc[feature_id],
            })

    combined = pd.concat(matrices, axis=1)
    combined.index = DISPLAY_TIME_LABELS
    metadata = pd.DataFrame(feature_rows)
    metadata.insert(0, "plot_position", np.arange(1, len(metadata) + 1))
    if combined.shape[1] != len(metadata):
        raise AssertionError("Matrix and feature metadata are out of sync")
    return combined, metadata, duplicate_rows_removed

def draw_heatmap(
    matrix, metadata, enzyme, duplicate_rows_removed, show_time_labels=True,
    output_stem=None, show_fraction_bar=False, show_all_labels=True,
    page_width_mm=190.0, page_height_mm=80.0, plot_bottom_mm=17.0,
    zscore_bar_size_mm=(24.0, 1.4),
):
    """Draw one fixed-size, steroid-blocked heatmap."""
    arial_path = Path("/System/Library/Fonts/Supplemental/Arial.ttf")
    if not arial_path.is_file():
        raise FileNotFoundError("Arial.ttf is required for this figure")
    fontManager.addfont(str(arial_path))
    arial_7 = FontProperties(fname=str(arial_path), size=7)
    arial_8 = FontProperties(fname=str(arial_path), size=8)
    arial_9 = FontProperties(fname=str(arial_path), size=9)

    plot_left_mm = 17.0
    plot_width_mm, plot_height_mm = page_width_mm - 20.5, 47.0
    tile_width, tile_height = 1.0, 1.0
    feature_gap, time_gap = 0.10, 0.16
    steroid_gap, condition_gap = 1.35, 0.75
    frame_padding = 0.25
    frame_color = "#999999"

    def box(left, bottom, width, height):
        return [
            left / page_width_mm,
            bottom / page_height_mm,
            width / page_width_mm,
            height / page_height_mm,
        ]

    x_centers = []
    group_spans = []
    cursor = 0.0
    for steroid in STEROID_ORDER:
        count = int(metadata["steroid"].eq(steroid).sum())
        positions = cursor + np.arange(count) * (tile_width + feature_gap)
        start = float(positions[0])
        end = float(positions[-1] + tile_width)
        x_centers.extend(positions + tile_width / 2)
        group_spans.append((steroid, start, end))
        cursor = end + feature_gap + steroid_gap
    matrix_width = group_spans[-1][2]

    y_centers = []
    y_cursor = 0.0
    for row_index in range(14):
        y_centers.append(y_cursor + tile_height / 2)
        y_cursor += tile_height
        if row_index < 13:
            y_cursor += condition_gap if row_index == 6 else time_gap
    matrix_height = y_cursor

    with plt.rc_context({
        "font.family": "Arial",
        "font.sans-serif": ["Arial"],
        "font.size": 8,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
        "axes.unicode_minus": True,
    }):
        fig = plt.figure(
            figsize=(page_width_mm / 25.4, page_height_mm / 25.4),
            facecolor="white",
        )
        axis = fig.add_axes(box(
            plot_left_mm, plot_bottom_mm, plot_width_mm, plot_height_mm,
        ))
        color_map = sns.color_palette("vlag", as_cmap=True)
        normalization = Normalize(vmin=-2.5, vmax=2.5, clip=True)
        values = matrix.to_numpy(dtype=float)

        # No edge stroke: spacing alone separates adjacent feature/time tiles.
        for column, x_center in enumerate(x_centers):
            for row, y_center in enumerate(y_centers):
                axis.add_patch(Rectangle(
                    (x_center - tile_width / 2, y_center - tile_height / 2),
                    tile_width, tile_height,
                    facecolor=color_map(normalization(values[row, column])),
                    edgecolor="none", linewidth=0, zorder=2,
                ))

        for previous, following in zip(group_spans[:-1], group_spans[1:]):
            axis.axvline(
                (previous[2] + following[1]) / 2,
                color="#AAAAAA", linewidth=0.65,
                linestyle=(0, (2.0, 2.5)), zorder=1,
            )

        axis.set_xlim(-frame_padding, matrix_width + frame_padding)
        axis.set_ylim(matrix_height + frame_padding, -frame_padding)
        axis.set_xticks([])
        time_labels = (
            DISPLAY_TIME_LABELS if show_all_labels and show_time_labels
            else [""] * 14
        )
        axis.set_yticks(y_centers, labels=time_labels)
        plt.setp(axis.get_yticklabels(), fontproperties=arial_7)
        axis.tick_params(
            axis="y", left=True, right=False, length=2.0, width=0.5,
            pad=2.5, color=frame_color, labelcolor="black",
        )
        for spine in axis.spines.values():
            spine.set_visible(True)
            spine.set_color(frame_color)
            spine.set_linewidth(0.65)

        for steroid, start, end in group_spans:
            axis.add_patch(Rectangle(
                (start, 1.018), end - start, 0.018,
                transform=axis.get_xaxis_transform(),
                facecolor=STEROID_COLORS[steroid],
                edgecolor="none", linewidth=0,
                clip_on=False, zorder=5,
            ))
            if show_all_labels:
                axis.annotate(
                    steroid,
                    xy=((start + end) / 2, 1),
                    xycoords=axis.get_xaxis_transform(),
                    xytext=(0, 8.0), textcoords="offset points",
                    ha="center", va="bottom",
                    fontproperties=arial_9, color="#377DA5",
                    annotation_clip=False,
                )

        if show_fraction_bar:
            # The plotted feature order is pellets then supernatant within each
            # steroid. Keep the strip on the same x scale as the heatmap tiles.
            fraction_names = []
            for source_groups in metadata["source_candidate_group"]:
                names = {
                    group.rsplit("_", 1)[-1]
                    for group in str(source_groups).split(";")
                }
                if len(names) != 1 or not names.issubset(FRACTION_COLORS):
                    raise ValueError(f"Cannot assign one fraction to {source_groups}")
                fraction_names.append(names.pop())

            strip_height_mm, strip_gap_mm = 1.0, 0.25
            strip_axis = fig.add_axes(box(
                plot_left_mm,
                plot_bottom_mm - strip_gap_mm - strip_height_mm,
                plot_width_mm, strip_height_mm,
            ))
            strip_axis.set_xlim(-frame_padding, matrix_width + frame_padding)
            strip_axis.set_ylim(0, 1)
            strip_axis.axis("off")
            run_start = 0
            for index in range(1, len(metadata) + 1):
                if index == len(metadata):
                    boundary = True
                else:
                    boundary = (
                        metadata.iloc[index]["steroid"]
                        != metadata.iloc[run_start]["steroid"]
                        or fraction_names[index] != fraction_names[run_start]
                    )
                if not boundary:
                    continue
                left = x_centers[run_start] - tile_width / 2
                right = x_centers[index - 1] + tile_width / 2
                strip_axis.add_patch(Rectangle(
                    (left, 0), right - left, 1,
                    facecolor=FRACTION_COLORS[fraction_names[run_start]],
                    edgecolor="none", linewidth=0,
                ))
                run_start = index

            fraction_legend_axis = fig.add_axes(box(
                plot_left_mm + plot_width_mm - 50.0, 2.8, 50.0, 7.0,
            ))
            fraction_legend_axis.set_xlim(0, 50.0)
            fraction_legend_axis.set_ylim(0, 7.0)
            fraction_legend_axis.axis("off")
            for fraction, left in (("supernatant", 0.0), ("pellets", 28.0)):
                fraction_legend_axis.add_patch(Rectangle(
                    (left, 2.0), 2.5, 2.5,
                    facecolor=FRACTION_COLORS[fraction],
                    edgecolor="none", linewidth=0,
                ))
                if show_all_labels:
                    fraction_legend_axis.text(
                        left + 3.3, 3.25, fraction.capitalize(),
                        ha="left", va="center", fontproperties=arial_8,
                    )

        colorbar_width_mm, colorbar_height_mm = zscore_bar_size_mm
        colorbar_axis = fig.add_axes(box(
            (page_width_mm - colorbar_width_mm) / 2,
            4.0,
            colorbar_width_mm,
            colorbar_height_mm,
        ))
        colorbar = fig.colorbar(
            ScalarMappable(norm=normalization, cmap=color_map),
            cax=colorbar_axis, orientation="horizontal",
            ticks=[-2.5, 0, 2.5],
        )
        if show_all_labels:
            colorbar.set_label("Row z-score", fontproperties=arial_8, labelpad=2)
            colorbar.ax.xaxis.set_label_position("top")
        colorbar.ax.tick_params(
            labelsize=8, length=2, width=0.5, pad=2,
            labelbottom=show_all_labels,
        )
        if show_all_labels:
            plt.setp(colorbar.ax.get_xticklabels(), fontproperties=arial_8)
        colorbar.outline.set_edgecolor(frame_color)
        colorbar.outline.set_linewidth(0.5)

        RESULTS.mkdir(parents=True, exist_ok=True)
        stem = output_stem or f"timecourse_CYP{enzyme}_human_microsome_style"
        if not show_all_labels:
            stem += "_no_labels"
        elif not show_time_labels:
            stem += "_no_time_labels"
        pdf_path = RESULTS / f"{stem}.pdf"
        svg_path = RESULTS / f"{stem}.svg"
        fig.savefig(pdf_path, dpi=300, facecolor="white")
        fig.savefig(svg_path, dpi=300, facecolor="white")
        force_svg_arial(svg_path)

    print(
        f"CYP{enzyme}: {matrix.shape[1]} steroid-feature columns, "
        f"{matrix.shape[0]} time points, {duplicate_rows_removed} duplicate rows removed"
    )
    print(pdf_path)
    print(svg_path)
    return fig

# Retain each matrix and its plotted-feature order for the concise 02 table.
figure_matrices = {}
figure_metadata = {}
RESULTS.mkdir(parents=True, exist_ok=True)
for enzyme in ENZYME_ORDER:
    matrix, metadata, duplicate_rows_removed = build_enzyme_matrix(tables, enzyme)
    observed = metadata.groupby("steroid", sort=False).size().to_dict()
    if set(observed) != set(STEROID_ORDER) or min(observed.values()) <= 0:
        raise AssertionError(f"Missing CYP{enzyme} steroid block: {observed}")
    full_enzyme = f"CYP{enzyme}"
    figure_matrices[full_enzyme] = matrix
    figure_metadata[full_enzyme] = metadata
    print(f"{full_enzyme} plotted features by steroid: {observed}")
    for show_time_labels, show_all_labels in ((True, True), (False, False)):
        figure = draw_heatmap(
            matrix, metadata, enzyme, duplicate_rows_removed,
            show_time_labels=show_time_labels,
            output_stem=f"timecourse_{full_enzyme}_03_network",
            show_fraction_bar=True,
            show_all_labels=show_all_labels,
            page_width_mm=175.0,
            page_height_mm=72.0,
            plot_bottom_mm=13.5,
            zscore_bar_size_mm=(22.25, 2.0),
        )
        plt.close(figure)



CYP3A4 plotted features by steroid: {'Estradiol': 113, 'DHEA': 71, 'Pregnenolone': 21}


CYP3A4: 205 steroid-feature columns, 14 time points, 0 duplicate rows removed
/Users/zhengjie/Desktop/P450_analysis/Figure_modification/Figure_2c_time_course/results/timecourse_CYP3A4_03_network.pdf
/Users/zhengjie/Desktop/P450_analysis/Figure_modification/Figure_2c_time_course/results/timecourse_CYP3A4_03_network.svg


CYP3A4: 205 steroid-feature columns, 14 time points, 0 duplicate rows removed
/Users/zhengjie/Desktop/P450_analysis/Figure_modification/Figure_2c_time_course/results/timecourse_CYP3A4_03_network_no_labels.pdf
/Users/zhengjie/Desktop/P450_analysis/Figure_modification/Figure_2c_time_course/results/timecourse_CYP3A4_03_network_no_labels.svg
CYP3A5 plotted features by steroid: {'Estradiol': 142, 'DHEA': 114, 'Pregnenolone': 60}


CYP3A5: 316 steroid-feature columns, 14 time points, 0 duplicate rows removed
/Users/zhengjie/Desktop/P450_analysis/Figure_modification/Figure_2c_time_course/results/timecourse_CYP3A5_03_network.pdf
/Users/zhengjie/Desktop/P450_analysis/Figure_modification/Figure_2c_time_course/results/timecourse_CYP3A5_03_network.svg


CYP3A5: 316 steroid-feature columns, 14 time points, 0 duplicate rows removed
/Users/zhengjie/Desktop/P450_analysis/Figure_modification/Figure_2c_time_course/results/timecourse_CYP3A5_03_network_no_labels.pdf
/Users/zhengjie/Desktop/P450_analysis/Figure_modification/Figure_2c_time_course/results/timecourse_CYP3A5_03_network_no_labels.svg


## 4. Export three concise tables

In [5]:
# Export only the three tables needed to review matching and reproduce the panels.
match_columns = [
    "steroid", "derivatization", "matrix", "enzyme", "source_feature_id",
    "unlabeled_feature_mz", "unlabeled_feature_rtime", "timecourse_feature_id",
    "timecourse_mz", "timecourse_rt", "ppm_difference", "rt_difference_s",
    "match_count", "match_rank", "match_status",
]
match_table = all_matches[match_columns].rename(columns={
    "unlabeled_feature_mz": "source_mz",
    "unlabeled_feature_rtime": "source_rt_s",
    "timecourse_rt": "timecourse_rt_s",
})
if len(match_table) != 721:
    raise AssertionError("Unexpected candidate-match count")
match_table.to_csv(RESULTS / "01_candidate_matches.tsv", sep="\t", index=False)

# Position comes from the exact matrices passed to the plotting function.
plot_order = pd.concat([
    metadata[["plot_position", "steroid", "timecourse_feature_id"]].assign(enzyme=enzyme)
    for enzyme, metadata in figure_metadata.items()
], ignore_index=True)
key = ["enzyme", "steroid", "timecourse_feature_id"]
if plot_order.duplicated(key).any() or len(plot_order) != 521:
    raise AssertionError("Plot positions must identify 521 enzyme/steroid features")
timecourse = integrated.merge(
    plot_order, on=key, how="left", sort=False, validate="many_to_one"
)
if len(timecourse) != 7294 or timecourse.plot_position.isna().any():
    raise AssertionError("A plotted time-course row lacks its plot position")
timecourse["_enzyme_order"] = timecourse.enzyme.map({e: i for i, e in enumerate(ENZYMES)})
timecourse["_condition_order"] = timecourse.condition.map({c: i for i, c in enumerate(CONDITIONS)})
timecourse = timecourse.sort_values(
    ["_enzyme_order", "plot_position", "_condition_order", "time_min"], kind="stable"
)
timecourse_columns = [
    "enzyme", "plot_position", "steroid", "derivatization", "matrix",
    "matched_source_feature_ids", "timecourse_feature_id", "condition",
    "time_min", "mean_normalized_log2", "row_zscore",
]
timecourse = timecourse[timecourse_columns]
if timecourse.groupby(["enzyme", "plot_position"]).size().ne(14).any():
    raise AssertionError("Every plotted feature must have 14 displayed time points")
timecourse.to_csv(RESULTS / "02_matched_timecourse.tsv", sep="\t", index=False)

processing_summary = summary.merge(
    normalization_summary, on=["steroid", "matrix"],
    how="left", validate="many_to_one"
)
summary_columns = [
    "steroid", "matrix", "enzyme", "candidate_count", "unique", "ambiguous",
    "unmatched", "plotted_timecourse_features", "n_features", "scale_min", "scale_max",
]
processing_summary = processing_summary[summary_columns].rename(columns={
    "plotted_timecourse_features": "plotted_feature_count",
    "n_features": "input_feature_count",
    "scale_min": "normalization_scale_min",
    "scale_max": "normalization_scale_max",
})
if len(processing_summary) != 12 or processing_summary.isna().any().any():
    raise AssertionError("Unexpected processing summary")
processing_summary.to_csv(RESULTS / "03_processing_summary.tsv", sep="\t", index=False)
print(f"Wrote 3 TSVs: {len(match_table)} matches, {len(timecourse)} plotted rows, {len(processing_summary)} groups")


Wrote 3 TSVs: 721 matches, 7294 plotted rows, 12 groups
